# பாடம் 04 - கருவி பயன்பாடு வடிவமைப்பு மாதிரி

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework (Python) பயன்படுத்தி AI முகவர்களுக்கான **கருவி பயன்பாடு** வடிவமைப்பு மாதிரியை கற்கப்போகிறீர்கள். இதில் உள்ளடக்கம்:

- `@tool` அலங்காரியுடன் மற்றும் typed அளவுருக்களுடன் செயல்பாட்டு கருவிகளை வரையறுத்தல்
- மாதிரி ஒவ்வொரு கருவியும் என்ன செய்யிறது என அறிய கருவி வார்ப்புருக்களை வழங்குதல்
- `approval_mode` கொண்டு கருவி செயல்பாட்டை கட்டுப்படுத்துதல்
- Pydantic மாதிரிகள் மற்றும் `response_format` மூலம் **இடுக்கப்பட்ட வெளியீட்டை** வழங்குதல்

காட்சி ஒரு **பயண முன்பதிவு முகவர்** ஆகும், இது இடங்களை கண்டறிய, கிடைக்கும்தன்மையைச் சரிபார்க்க மற்றும் விமான தகவலைப் பெற உதவுகிறது.


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool அலங்காரியைப் பயன்படுத்தி கருவிகளை வரையறுத்தல்

`@tool` அலங்காரி ஒரு சாதாரண Python செயல்பாட்டை, ஒரு முகவர் அழைக்கக்கூடிய கருவியாக மாற்றுகிறது.
முக்கியக் குறிப்புகள்:

- **docstring** என்பது மாதிரி பார்க்கும் கருவி விளக்கமாக மாறும்.
- **தேவைப்பட்டிருப்பு வகை குறிப்புகள்** (`Annotated` உட்பட விளக்கங்களுடன்) கருவி வடிவத்தை வரையறுக்கிறது.
- `approval_mode` என்பது ஒவ்வொரு அழைப்பையும் செயல்படுத்தும் முன் பயனர் உறுதிப்படுத்த வேண்டுமா என்பதை கட்டுப்படுத்தும்.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## பல கருவிகளைக் கொண்டு ஒரு முகவரையை உருவாக்கல்

பயனரின் கேள்விக்கு பதில் அளிக்க மாடல் தேவையாயின் எந்த கருவிகளையும் அழைக்க, மூன்று கருவிகளையும் கிளையன்டுக்கு அனுப்புக.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## கருவிகளுடன் வடிவமைக்கப்பட்ட வெளியீடு

`response_format` ஐ Pydantic மாடலைக் கொண்டு அமைத்தால், முகவர் விடுதலை செய்யும் தரவு ஒரு நன்கு வகைப்படுத்தப்பட்ட JSON பொருள் ஆக இருக்க வேண்டும், சுதந்திர வடிவ உரை அல்ல. இது வினாமாக குறியிடப்பட்ட கோடுகள் முடிவுகளை நிரலாக்க முறையில் பயன்படுத்தும் போது பயனுள்ளதாக இருக்கும்.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## கருவி அனுமதி முறைகள்

`@tool` ஐல் உள்ள `approval_mode` மாற்றி கருவி அழைப்புகள் செயல்படும்முன் மனித அனுமதி தேவையா என்பதை கட்டுப்படுத்துகிறது:

| முறை | நடத்தைகள் |
|---|---|
| `"never_require"` | கருவி தானாக இயங்கும் — பயனர் உறுதிப்படுத்தல் தேவையில்லை. |
| `"always_require"` | ஒவ்வொரு அழைப்பும் செயல்படும்முன் பயனர் மூலம் அங்கீகரிக்கப்பட வேண்டும். |

பக்கவிளைவுகள் உண்டாக்கும் கருவிகளுக்கு (எ.கா. விமானம் முன்பதிவு, கடன் அட்டை சார்ஜ் செய்யுதல்) `"always_require"` ஐ பயன்படுத்தவும், இதனால் மனிதர் சுழற்சியில் நின்று கவனிக்க முடியும்.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் எப்படி செய்வது என்பதை கற்றுக்கொண்டீர்கள்:

1. **கருவிகள்** என்பதை `@tool` அலங்காரம் மூலம் வகைப்படுத்தப்பட்ட அளவுருக்கள் மற்றும் கருவி ஸ்கீமாவாக செயல்படும் டாக்ஸ்ட்ரிங்களுடன் வரையறுக்கவும்.
2. **பல கருவிகளை உருவாக்குக** என்பதன் மூலம் சார்பாளர்கள் சிக்கலான கேள்விகளுக்கு பதிலளிக்க அவற்றை வரிசையாக அழைக்க முடியும்.
3. **கட்டமைக்கப்பட்ட வெளியீட்டை** Pydantic மodelஐ `response_format` என கடந்துவைத்து வழங்கவும்.
4. **கருவி அனுமதியை கட்டுப்படுத்தவும்** `approval_mode` மூலம் உணர்ச்சிபூர்வ செயல்களுக்கு மனிதர் கட்டுப்பாட்டில் இருக்கும்படி.

இந்த முறைமைகளை நம்பகமான, தயாரிப்புக்குத் தயாரான சார்பாளர்களை உருவாக்குவதற்கான அடித்தளமாகக் கருதலாம், அவை வெளிப்புற அமைப்புகளுடன் பாதுகாப்பாக தொடர்பு கொள்கின்றன.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
